In [ ]:
'''
This script takes the processed 2-photon tif files as well as the semgmention
masks for those files and can either do transfer learning based on the DeepD3
weights or from scratch model training. From scratch training compresses a
designated number of slices into a pseudo-plane to create 2.5D data. Then uses
that data to train a Unet model for the segmentation of spines and dendrites in
each slice. The transfer learning only uses normal 2D slices.

Inputs
- Folder of processed .tifs of 2-photon images
- Segmentation masks for spines and dendrites of each of those .tifs

Outputs
- Trained Unet model for segmenting spines as a .keras file
'''

import os
import random
import threading
import queue
import time
from pathlib import Path

import numpy as np
import tensorflow as tf
import keras
from keras import layers, Model
from tqdm import tqdm

from scipy.ndimage import binary_dilation, zoom
from scipy.optimize import linear_sum_assignment
from scipy.spatial import distance_matrix
from skimage import io, measure
from skimage.measure import regionprops, label
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# -------------- CONFIGURATION --------------

# ------- Training Mode -------
# can be set to transfer or scratch depending on if you want to do
# transfer learning on DeepD3 or train a model from scratch
TRAINING_MODE = 'scratch'

# ------- For Both Modes -------
# Input the resolultion of your image
YOUR_XY_UM = 0.096  # um/pixel xy
YOUR_Z_UM = 1.0    # um/slice

# File path names and files names for inputs and outputs
# BASE_DIR = '/content/drive/MyDrive/tiffstacks'
# OUTPUT_DIR = f'{BASE_DIR}/results'
BASE_DIR = Path.cwd()
TRAIN_DIR = BASE_DIR / 'Inputs' / 'traindata'
OUTPUT_DIR = BASE_DIR.parent / 'Output'

# Scratch mode saves here; transfer mode loads from PRETRAINED_WEIGHTS and saves here
SCRATCH_MODEL      = OUTPUT_DIR / 'trained_models' / 'deepd3_scratchmodel.keras'
FINETUNED_MODEL    = OUTPUT_DIR / 'trained_models' / 'deepd3_finetuned.keras'

OUTPUT_MODEL = SCRATCH_MODEL if TRAINING_MODE == 'scratch' else FINETUNED_MODEL
OUTPUT_MODEL.parent.mkdir(exist_ok=True, parents=True)

TRAIN_STACKS = [TRAIN_DIR / 'G40_raw.tif',
                TRAIN_DIR / 'F14_raw.tif',
                TRAIN_DIR / '20250323_roi1_raw.tif',
                TRAIN_DIR / '20250325_roi1_raw.tif']
TRAIN_DENDRITES = [TRAIN_DIR / 'G40_dendrite.tif',
                   TRAIN_DIR / 'F14_dendrite.tif',
                   TRAIN_DIR / '20250323_roi1_dendrite.tif',
                   TRAIN_DIR / '20250325_roi1_dendrite.tif']
TRAIN_SPINES = [TRAIN_DIR / 'G40_spine.tif',
                TRAIN_DIR / 'F14_spine.tif',
                TRAIN_DIR / '20250323_roi1_spine.tif',
                TRAIN_DIR / '20250325_roi1_spine.tif']

INFERENCE_STACK = TRAIN_DIR / '20240817_roi3_raw.tif'
TRUE_DENDRITE = TRAIN_DIR / '20240817_roi3_dendrite.tif'
TRUE_SPINE = TRAIN_DIR / '20240817_roi3_spine.tif'

assert INFERENCE_STACK.exists()

# Parameters for model training
EPOCHS = 50
STEPS = 200
BATCH_SIZE = 4
VAL_BATCHES = 25

INITIAL_LR = 1e-4
LR_PATIENCE = 5
LR_FACTOR = 0.2
MIN_DELTA = 0.001  # min improvement to reset patience
EMA_SMOOTHING = 0.3

# ------- For Scratch Mode -------
# Specifies the numbner of slices you want to compress into 1 layer to
# create the 2.5 data used in training. (includes center so must be odd)
STACK_COMPRESS = 7

# ------- For Transfer Mode -------
# Load the weights of the DeepD3 model
PRETRAINED_WEIGHTS = BASE_DIR / 'Inputs' / 'weights' / 'DeepD3_16F_94nm.h5'
assert PRETRAINED_WEIGHTS.exists()

# Zoom input data to match DeepD3 training data resolution
TARGET_XY_UM = 0.094
TARGET_Z_UM  = 0.5
ZOOM_Z  = YOUR_Z_UM  / TARGET_Z_UM
ZOOM_XY = YOUR_XY_UM / TARGET_XY_UM

# Lays out the layer unfreezing schedule for transfer learning mode
# the first number is epoch to unfreeze at the second is what stage of the
# encoder to unfreeze and the third is a new learning rate.
UNFREEZE_SCHEDULE = {
    10: (3, 5e-5),
    20: (2, 2e-5),
    30: (1, 1e-5),
    40: (0, 5e-6),
}

# ---------------------------------------------------

In [ ]:
# Define the model architecture

# Define the encoder block structure
def residual_enc_block(x, filters, block_name):
    out = layers.Conv2D(filters, 3, padding='same', use_bias=False, name=f'{block_name}_conv1')(x)
    out = layers.BatchNormalization(name=f'{block_name}_conv1_BN')(out)
    out = layers.Activation('swish', name=f'{block_name}_conv1_activation')(out)
    out = layers.Conv2D(filters, 3, padding='same', use_bias=False, name=f'{block_name}_conv2')(out)

    identity = layers.Conv2D(filters, 1, padding='same', use_bias=False, name=f'{block_name}_identity')(x)

    out = layers.BatchNormalization(name=f'{block_name}_conv2_BN')(out)
    out = layers.Add(name=f'{block_name}_conv2_residual_connection')([identity, out])
    out = layers.Activation('swish', name=f'{block_name}_conv2_activation')(out)
    return out

# Define the decoder block structure
def decoder_block(x, skip, filters, block_name):
    x = layers.UpSampling2D()(x)
    x = layers.Concatenate()([x, skip])
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False, name=f'{block_name}_conv1')(x)
    x = layers.BatchNormalization(name=f'{block_name}_conv1_BN')(x)
    x = layers.Activation('swish', name=f'{block_name}_conv1_activation')(x)
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False, name=f'{block_name}_conv2')(x)
    x = layers.BatchNormalization(name=f'{block_name}_conv2_BN')(x)
    x = layers.Activation('swish', name=f'{block_name}_conv2_activation')(x)
    return x

# Build the model of one encoder with a dual decoder one for spines and one
# for dendrites
def build_deepd3(input_shape=(None, None, 1)):
    inputs = layers.Input(shape=input_shape, name='input')

    e0 = residual_enc_block(inputs, 16, 'enc_layer0')
    p0 = layers.MaxPooling2D()(e0)
    e1 = residual_enc_block(p0, 32, 'enc_layer1')
    p1 = layers.MaxPooling2D()(e1)
    e2 = residual_enc_block(p1, 64, 'enc_layer2')
    p2 = layers.MaxPooling2D()(e2)
    e3 = residual_enc_block(p2, 128, 'enc_layer3')
    p3 = layers.MaxPooling2D()(e3)

    latent = layers.Conv2D(256, 3, padding='same', use_bias=False, name='latent_conv')(p3)
    latent = layers.BatchNormalization(name='latent_conv_BN')(latent)
    latent = layers.Activation('swish', name='latent_conv_activation')(latent)

    dd4 = decoder_block(latent, e3, 128, 'dendrites_dec_layer4')
    dd3 = decoder_block(dd4, e2, 64, 'dendrites_dec_layer3')
    dd2 = decoder_block(dd3, e1, 32, 'dendrites_dec_layer2')
    dd1 = decoder_block(dd2, e0, 16, 'dendrites_dec_layer1')

    ds4 = decoder_block(latent, e3, 128, 'spines_dec_layer4')
    ds3 = decoder_block(ds4, e2, 64, 'spines_dec_layer3')
    ds2 = decoder_block(ds3, e1, 32, 'spines_dec_layer2')
    ds1 = decoder_block(ds2, e0, 16, 'spines_dec_layer1')

    dendrites_out = layers.Conv2D(1, 1, activation='sigmoid', use_bias=True, name='dendrites')(dd1)
    spines_out = layers.Conv2D(1, 1, activation='sigmoid', use_bias=True, name='spines')(ds1)

    return Model(inputs=inputs, outputs=[dendrites_out, spines_out], name='DeepD3')

In [ ]:
# ------- LOSS FUNCTIONS -------

def dice_loss(y_true, y_pred, smooth=1e-5):
    """
    Calculates dice loss, which is basically a measure of how well two
    the two segmentations overlap.
    """
    y_true = tf.cast(y_true, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    return 1 - (2 * intersection + smooth) / (
        tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)


def batch_dice(y_true, y_pred, smooth=1e-5):
    """
    Calcualtes mean dice score across batches to be used as a training
    accuracy metric.
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    union = tf.reduce_sum(y_true, axis=[1, 2, 3]) + tf.reduce_sum(y_pred, axis=[1, 2, 3])
    return tf.reduce_mean((2 * intersection + smooth) / (union + smooth)).numpy()


def tversky_loss(y_true, y_pred, alpha=0.55, beta=0.45, smooth=1e-5):
    """
    Calcualtes Tversky loss, a generalization of dice that lets you set the penalty
    for false positives (alpha) and false negatives (beta) independently.
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    tp = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    fp = tf.reduce_sum((1.0 - y_true) * y_pred, axis=[1, 2, 3])
    fn = tf.reduce_sum(y_true * (1.0 - y_pred), axis=[1, 2, 3])
    tversky_index = (tp + smooth) / (tp + alpha * fp + beta * fn + smooth)
    return tf.reduce_mean(1.0 - tversky_index)


def combined_loss(y_true, y_pred, alpha=0.55, beta=0.45):
    """
    Calcualtes a combined loss metric of binary crossentropy (BCE) and Tversky
    loss. BCE for pixel-level stability and Tversky loss for class imbalance.
    BCE handles the easy background pixels while Tversky focuses the main pixels
    of interest. (spines and dendrites)
    """
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    bce_loss = tf.reduce_mean(tf.expand_dims(bce, axis=-1))
    return bce_loss + tversky_loss(y_true, y_pred, alpha=alpha, beta=beta)

In [ ]:
# ------- DATA PIPELINE -------

def precompute_annotated_slices(stacks, dendrites, spines):
    """
    Scans all stacks and records slices of images that contain enough
    annotationed pixels to sample from.
    Thresholds: >=50 dendrite pixels OR >=20 spine pixels.
    """
    annotated_slices = []
    for idx in range(len(stacks)):
        for z in range(stacks[idx].shape[0]):
            if dendrites[idx][z].sum() >= 50 or spines[idx][z].sum() >= 20:
                annotated_slices.append((idx, z))
    return annotated_slices


def get_random_tile(stacks, dendrites, spines, annotated_slices,
                    tile_size=128, max_attempts=50):
    """
    Samples a random 128×128 tile from a slice and in scratch mode builds a
    pseudo-plane by stacking STACK_COMPRESS/2 adjacent slices on each side
    along the channel axis to create 2.5D data, does not do this in transfer
    mode. Retries up to max_attempts to find a tile with at least 10 dendrite
    or 5 spine pixels.
    """
    offset = STACK_COMPRESS // 2

    for _ in range(max_attempts):
        idx, z = random.choice(annotated_slices)
        Z, H, W = stacks[idx].shape
        y = random.randint(0, H - tile_size)
        x = random.randint(0, W - tile_size)

        if TRAINING_MODE == 'scratch':
            tile = np.stack(
                [stacks[idx][np.clip(zi, 0, Z - 1), y:y+tile_size, x:x+tile_size]
                 for zi in range(z - offset, z + offset + 1)],
                axis=-1
            )
        else:
            tile = stacks[idx][z, y:y+tile_size, x:x+tile_size][..., np.newaxis]

        d_lbl = dendrites[idx][z, y:y+tile_size, x:x+tile_size]
        s_lbl = spines[idx][z, y:y+tile_size, x:x+tile_size]

        if d_lbl.sum() >= 10 or s_lbl.sum() >= 5:
            return augment_tile(normalize_tile(tile), d_lbl, s_lbl)

    return augment_tile(normalize_tile(tile), d_lbl, s_lbl)

def normalize_tile(tile):
    """
    Scales all pixels in a tile to be between -1 and 1.
    """
    t = tile.astype(np.float32)
    return ((t - t.min()) / (t.max() - t.min() + 1e-5)) * 2 - 1


def augment_tile(tile, dendrite, spine):
    """
    Applies aumgentation to data including random rotations, flips,
    brightness jitters, and Gaussian noise.
    """
    k = random.randint(0, 3)
    tile, dendrite, spine = np.rot90(tile, k), np.rot90(dendrite, k), np.rot90(spine, k)
    if random.random() > 0.5:
        tile, dendrite, spine = np.fliplr(tile), np.fliplr(dendrite), np.fliplr(spine)
    if random.random() > 0.5:
        tile, dendrite, spine = np.flipud(tile), np.flipud(dendrite), np.flipud(spine)
    if random.random() > 0.5:
        tile = np.clip(tile * random.uniform(0.8, 1.2), -1, 1)
    if random.random() > 0.5:
        tile = np.clip(tile + np.random.normal(0, 0.05, tile.shape).astype(np.float32), -1, 1)
    return tile, dendrite, spine


def get_batch(stacks, dendrites, spines, annotated_slices, batch_size=8, tile_size=128):
    """
    Builds a batch by calling the function get_random_tile() batch_size times
    and adds a dimesion to the masks to match the model output.
    """
    imgs, d_lbls, s_lbls = [], [], []
    for _ in range(batch_size):
        img, d, s = get_random_tile(stacks, dendrites, spines, annotated_slices, tile_size)
        imgs.append(img)
        d_lbls.append(d)
        s_lbls.append(s)
    imgs = np.array(imgs,   dtype=np.float32)
    d_lbls = np.array(d_lbls, dtype=np.float32)[..., np.newaxis]
    s_lbls = np.array(s_lbls, dtype=np.float32)[..., np.newaxis]
    return imgs, d_lbls, s_lbls


class DeepD3Dataset:
    """
    Loads tif stacks and label masks, utilizes precompute_annotated_slices from
    earlier, then serves batches during training. In transfer mode, resamples
    the stacks to the target resolution before loading.
    """
    def __init__(self, stack_paths, dendrite_paths, spine_paths, tile_size=128):
        self.tile_size = tile_size
        self.stacks, self.dendrites, self.spines = [], [], []

        for s_path, d_path, sp_path in zip(stack_paths, dendrite_paths, spine_paths):
            stack    = io.imread(s_path).astype(np.float32)
            dendrite = io.imread(d_path).astype(np.float32)
            spine    = io.imread(sp_path).astype(np.float32)

            if TRAINING_MODE == 'transfer':
                stack    = zoom(stack, (ZOOM_Z, ZOOM_XY, ZOOM_XY), order=1)
                dendrite = zoom(dendrite, (ZOOM_Z, ZOOM_XY, ZOOM_XY), order=0)
                spine    = zoom(spine, (ZOOM_Z, ZOOM_XY, ZOOM_XY), order=0)

            self.stacks.append(stack)
            self.dendrites.append((dendrite > 0).astype(np.float32))
            self.spines.append((spine > 0).astype(np.float32))

        self.annotated_slices = precompute_annotated_slices(
            self.stacks, self.dendrites, self.spines)

    def get_batch(self, batch_size=8):
        return get_batch(self.stacks, self.dendrites, self.spines,
                         self.annotated_slices, batch_size, self.tile_size)

class PrefetchDataset:
    """
    Makes CPU get batch in the background while GPU is training to help
    increase the speed of training.
    """
    def __init__(self, dataset, batch_size, prefetch_size=4):
        self.dataset = dataset
        self.batch_size = batch_size
        self.queue = queue.Queue(maxsize=prefetch_size)
        self.thread = threading.Thread(target=self._worker, daemon=True)
        self.thread.start()

    def _worker(self):
        while True:
            self.queue.put(self.dataset.get_batch(self.batch_size))

    def get_batch(self):
        return self.queue.get()

In [ ]:
# ------- INFERENCE AND POST PROCESSING -------

def predict_plane_tiled_batched(model, pseudo_plane, tile_size=128, inset_size=96, batch_size=64):
    """
    Tiles inference stack based on tiles size and runs inference on a single
    pseudo-plane of 2.5D data. Only the center inset_size x
    inset_size of each tile is kept to avoid edge artifacts. Tiles are then
    stitched back together for the output of the whole slice.
    """
    h, w, c = pseudo_plane.shape
    off = (tile_size - inset_size) // 2
    pad_h = int(np.ceil(h / inset_size) * inset_size)
    pad_w = int(np.ceil(w / inset_size) * inset_size)

    plane_padded = np.pad(pseudo_plane,
                          ((off, pad_h - h + off), (off, pad_w - w + off), (0, 0)),
                          mode='constant', constant_values=pseudo_plane.mean())

    tiles, coords = [], []
    for y in range(0, pad_h, inset_size):
        for x in range(0, pad_w, inset_size):
            tiles.append(normalize_tile(plane_padded[y:y+tile_size, x:x+tile_size, :]))
            coords.append((y, x))

    tiles = np.array(tiles, dtype=np.float32)

    d_preds, s_preds = [], []
    for i in range(0, len(tiles), batch_size):
        dp, sp = model(tiles[i:i+batch_size], training=False)
        d_preds.append(dp.numpy())
        s_preds.append(sp.numpy())

    d_preds = np.concatenate(d_preds, axis=0)
    s_preds = np.concatenate(s_preds, axis=0)

    d_full = np.zeros((pad_h, pad_w), dtype=np.float32)
    s_full = np.zeros((pad_h, pad_w), dtype=np.float32)
    for i, (y, x) in enumerate(coords):
        d_full[y:y+inset_size, x:x+inset_size] = d_preds[i, off:off+inset_size, off:off+inset_size, 0]
        s_full[y:y+inset_size, x:x+inset_size] = s_preds[i, off:off+inset_size, off:off+inset_size, 0]

    return d_full[:h, :w], s_full[:h, :w]


def run_stack_inference(model, tiff_path, threshold=0.5, tile_size=128, inset_size=96, batch_size=64):
    """
    Uses predict_plane_tiled_batched to run full tiled inference slice by slice
    on a whole 3D tif stack. Returns the original stack alongside dendrite and
    spine probability maps and binary masks thresholded based on the threshold
    value set. In transfer mode, resamples the data to the target resolution
    before running inference then resamples back to the original resolution
    after.
    """
    t0 = time.time()
    stack = io.imread(tiff_path).astype(np.float32)

    stack_rs = zoom(stack, (ZOOM_Z, ZOOM_XY, ZOOM_XY), order=1) if TRAINING_MODE == 'transfer' else stack
    Z, H, W = stack_rs.shape
    offset = STACK_COMPRESS // 2

    d_probs = np.zeros((Z, H, W), dtype=np.float32)
    s_probs = np.zeros((Z, H, W), dtype=np.float32)

    for z in tqdm(range(Z), desc="Z-Slices"):
        if TRAINING_MODE == 'scratch':
            indices = np.clip(np.arange(z - offset, z + offset + 1), 0, Z - 1)
            pseudo_plane = np.stack([stack_rs[i] for i in indices], axis=-1)
        else:
            pseudo_plane = stack_rs[z][..., np.newaxis]

        d_probs[z], s_probs[z] = predict_plane_tiled_batched(
            model, pseudo_plane, tile_size, inset_size, batch_size)

    if TRAINING_MODE == 'transfer':
        inv_z, inv_xy = 1.0 / ZOOM_Z, 1.0 / ZOOM_XY
        d_probs = np.clip(zoom(d_probs, (inv_z, inv_xy, inv_xy), order=1), 0, 1)
        s_probs = np.clip(zoom(s_probs, (inv_z, inv_xy, inv_xy), order=1), 0, 1)

    d_masks = (d_probs > threshold).astype(np.uint8)
    s_masks = (s_probs > threshold).astype(np.uint8)
    return stack, d_probs, s_probs, d_masks, s_masks


def filter_dendrite_voxels(mask_stack, min_voxels=500):
    """
    Filters dendrite inference by looking across multiple slices and connecting
    dendrites that touch and discarding any volume smaller than min_voxels.
    """
    labels = measure.label(mask_stack)
    component_sizes = np.bincount(labels.ravel())
    cleaned = mask_stack.copy()
    cleaned[component_sizes[labels] < min_voxels] = 0
    return cleaned


def filter_spines_by_dendrite_proximity(dendrite_masks, spine_masks,
                                        xy_distance_px=20, z_range=2):
    """
    Filters spine inference by checking if spine pixels are within
    xy_distance_px pixels circle and z_range slices of any dendrite voxel and
    removes any spines that are not withtin that area.
    """
    Z, H, W = spine_masks.shape
    filtered_spines = np.zeros_like(spine_masks)

    y_grid, x_grid = np.ogrid[-xy_distance_px:xy_distance_px+1,
                               -xy_distance_px:xy_distance_px+1]
    circle_kernel = (y_grid**2 + x_grid**2 <= xy_distance_px**2)

    for z in tqdm(range(Z), desc="Filtering spines"):
        z_min = max(0,   z - z_range)
        z_max = min(Z-1, z + z_range)
        dendrite_neighborhood = dendrite_masks[z_min:z_max+1].any(axis=0)
        search_region = binary_dilation(dendrite_neighborhood, structure=circle_kernel)
        filtered_spines[z] = (spine_masks[z] & search_region).astype(np.uint8)

    removed = spine_masks.sum() - filtered_spines.sum()
    print(f"------- Spine Filtering -------")
    print(f"Before:  {spine_masks.sum():,} px")
    print(f"After:   {filtered_spines.sum():,} px")
    print(f"Removed: {removed:,} px  ({removed/max(spine_masks.sum(),1)*100:.1f}%)")
    return filtered_spines

In [ ]:
# ------- EVALUATION AND VISUALIZATION -------

def evaluate_segmentation(pred_masks, true_masks, name=""):
    """
    Computes dice, precision, and recall between predicted and ground truth
    masks. and prints these metrics into a small descritpion for readablility.
    """
    pred = pred_masks.astype(bool).flatten()
    true = true_masks.astype(bool).flatten()
    tp = np.logical_and(pred,  true).sum()
    fp = np.logical_and(pred, ~true).sum()
    fn = np.logical_and(~pred, true).sum()
    smooth = 1e-7
    dice = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
    precision = (tp + smooth) / (tp + fp + smooth)
    recall = (tp + smooth) / (tp + fn + smooth)
    print(f"------- {name} -------")
    print(f"Dice:      {dice:.4f}   (1.0 = perfect overlap)")
    print(f"Precision: {precision:.4f}   (low = too many false positives)")
    print(f"Recall:    {recall:.4f}   (low = too many missed detections)")
    return {'dice': dice, 'precision': precision, 'recall': recall,
            'tp': int(tp), 'fp': int(fp), 'fn': int(fn)}


def evaluate_stack(d_masks, s_masks, true_dendrite_path, true_spine_path):
    """
    Loads ground truth tifs and runs utilizes the evaluate_segmentation function
    on both the spines and dendrites individually.
    """
    true_dendrite = (io.imread(true_dendrite_path) > 0).astype(np.uint8)
    true_spine = (io.imread(true_spine_path) > 0).astype(np.uint8)
    return {
        'dendrite': evaluate_segmentation(d_masks, true_dendrite, name="Dendrite"),
        'spine': evaluate_segmentation(s_masks, true_spine, name="Spine"),
    }


def evaluate_spine_centroids(y_true, y_pred, pixel_resolution=(1.0, 0.096, 0.096), dist_threshold=2.0):
    """
    Evaluateds spine segmentation by matching centroids between ground truth
    and predicted spine segmentations. Uses the Hungarian algorithm to find the
    optimal matching, then filters matches by dist_threshold (microns) so that
    spines too far away from each other are not matched. Returns precision,
    recall, and f1 score based on these matches.
    """
    gt_coords = [np.array(p.centroid) * pixel_resolution for p in regionprops(label(y_true))]
    pr_coords = [np.array(p.centroid) * pixel_resolution for p in regionprops(label(y_pred))]
    n_gt, n_pr = len(gt_coords), len(pr_coords)

    if n_gt == 0 or n_pr == 0:
        return {"precision": 0.0, "recall": 0.0, "f1_score": 0.0,
                "avg_dist_microns": np.nan}

    dist_mat = distance_matrix(gt_coords, pr_coords)
    gt_idx, pr_idx = linear_sum_assignment(dist_mat)

    matched_distances = [dist_mat[g, p] for g, p in zip(gt_idx, pr_idx)
                         if dist_mat[g, p] <= dist_threshold]
    tp = len(matched_distances)
    fp = n_pr - tp
    fn = n_gt - tp

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return {"precision": precision, "recall": recall, "f1_score": f1,
            "true_positives": tp, "false_positives": fp, "false_negatives": fn,
            "avg_dist_microns": np.mean(matched_distances) if matched_distances else np.nan}

def visualize_overlay(stack, d_masks, s_masks, overlay_slice):
    """
    Visualization of a single Z slice of segmentaion. Green is dendrite,
    red is spines, yellow is the overlap. Raw image shown as well for
    comparison.
    """
    Z = stack.shape[0]
    raw = stack[overlay_slice].astype(np.float32)
    raw = (raw - raw.min()) / (raw.max() - raw.min() + 1e-8)
    rgb = np.stack([raw, raw, raw], axis=-1)
    alpha = 0.5

    for mask, ch_on, ch_off in [(d_masks[overlay_slice].astype(bool), 1, [0, 2]),
                                 (s_masks[overlay_slice].astype(bool), 0, [1, 2])]:
        rgb[mask, ch_on] = rgb[mask, ch_on]  * (1 - alpha) + alpha
        for ch in ch_off:
            rgb[mask, ch] = rgb[mask, ch] * (1 - alpha)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f'Overlay — Z slice {overlay_slice}/{Z-1}', fontsize=13)
    axes[0].imshow(raw, cmap='gray'); axes[0].set_title('Raw Input')
    axes[1].imshow(np.clip(rgb, 0, 1)); axes[1].set_title('Overlay')
    axes[1].legend(handles=[Patch(color='green',  label='Dendrite'),
                             Patch(color='red',    label='Spine'),
                             Patch(color='yellow', label='Overlap')],
                             loc='lower right', fontsize=9)
    for ax in axes: ax.axis('off')
    plt.tight_layout(); plt.show()


def visualize_comparison(stack, d_masks, s_masks, true_d_path, true_s_path,
                          slice_idx=0, save_path=None):
    """
    Visualization of the comparison of ground truth vs model prediction for a
    single slice. Ground truth uses cyan for dendrite and magenta for spines
    the while model prediction uses green dendrite and red for spines.
    Optionally saves as a PDF if save path is specified.
    """
    gt_d = (io.imread(true_d_path)[slice_idx] > 0).astype(np.uint8)
    gt_s = (io.imread(true_s_path)[slice_idx] > 0).astype(np.uint8)
    raw = stack[slice_idx]

    def plot_overlay(ax, img, d_mask, s_mask, d_color, s_color, title):
        img_display = (img - img.min()) / (img.max() - img.min() + 1e-8)
        ax.imshow(img_display.astype(np.float32), cmap='gray')
        for mask, color, alpha in [(d_mask, d_color, 0.4), (s_mask, s_color, 0.6)]:
            overlay = np.zeros((*mask.shape, 4))
            overlay[mask > 0] = [*color, alpha]
            ax.imshow(overlay)
        ax.set_title(title, fontsize=16, fontweight='bold')
        ax.axis('off')

    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    plot_overlay(axes[0], raw, gt_d, gt_s, [0,1,1], [1,0,1], f"Ground Truth (Slice {slice_idx})")
    plot_overlay(axes[1], raw, d_masks[slice_idx], s_masks[slice_idx], [0,1,0], [1,0,0], f"Prediction (Slice {slice_idx})")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, format='pdf', dpi=300, bbox_inches='tight')
        print(f"Saved PDF to {save_path}")
    plt.show()

def plot_training_history(history, mode='scratch', unfreeze_schedule=None):
    """
    Plots training and validation loss and dice across epochs. In scratch mode,
    marks epochs where the learning rate was reduced. In transfer mode, marks
    epochs where an encoder stages are unfrozen.
    """
    epochs = range(1, len(history['train_loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Training History ({mode})', fontsize=13)

    # Loss plot
    axes[0].plot(epochs, history['train_loss'], label='Train Loss')
    axes[0].plot(epochs, history['val_loss'],   label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss')
    axes[0].legend()

    # Dice plot (accuracy)
    axes[1].plot(epochs, history['train_dice'], label='Train Dice')
    axes[1].plot(epochs, history['val_dice'],   label='Val Dice')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Dice')
    axes[1].set_title('Dice')
    axes[1].legend()

    # Adds markers for LR reductions (scratch) or unfreeze points (transfer)
    if mode == 'transfer' and unfreeze_schedule:
        for ep, (stage, _) in unfreeze_schedule.items():
            for ax in axes:
                ax.axvline(x=ep, color='red', linestyle='--', alpha=0.5)
                ax.text(ep + 0.2, ax.get_ylim()[1] * 0.95,
                        f'enc{stage}', fontsize=8, color='red')
    elif mode == 'scratch' and 'lr_reductions' in history:
        for ep in history['lr_reductions']:
            for ax in axes:
                ax.axvline(x=ep, color='orange', linestyle='--', alpha=0.5)
                ax.text(ep + 0.2, ax.get_ylim()[1] * 0.95,
                        'LR↓', fontsize=8, color='orange')

    plt.tight_layout()
    plt.show()

In [ ]:
# ------- TRANSFER LEARNING -------
def freeze_all_encoder(model):
    """
    Freezes all encoder and latent layers and leaves the decoder trainable.
    """
    for layer in model.layers:
        layer.trainable = not any(layer.name.startswith(p)
                                  for p in ['enc_layer', 'latent'])
    trainable = sum(np.prod(w.shape) for w in model.trainable_weights)
    non_trainable = sum(np.prod(w.shape) for w in model.non_trainable_weights)
    print(f"Trainable:     {trainable:,}")
    print(f"Non-trainable: {non_trainable:,}")
    return model


def unfreeze_encoder_stage(model, optimizer, stage, learning_rate):
    """
    Function to unfreeze one encoder stage and update the optimizer learning
    rate. Prints out these changes whenever the happen to the console.
    """
    for layer in model.layers:
        if layer.name.startswith(f'enc_layer{stage}') or \
           (stage == 0 and layer.name.startswith('latent')):
            layer.trainable = True

    trainable = sum(np.prod(w.shape) for w in model.trainable_weights)
    print(f"  Unfroze enc_layer{stage}{' + latent' if stage == 0 else ''} "
          f"— trainable params: {trainable:,}  LR now {learning_rate:.0e}")

In [ ]:
# Build model and define the input shape which depends on if you are using
# scratch mode which uses 2.5D data or transer which just used 2D
input_channels = STACK_COMPRESS if TRAINING_MODE == 'scratch' else 1
model = build_deepd3(input_shape=(None, None, input_channels))

if TRAINING_MODE == 'transfer':
    # Load pretrained weights from DeepD3 into defined model and skips layers
    # that do not have matching names and reports any skipped layers
    pretrained = tf.keras.models.load_model(PRETRAINED_WEIGHTS, compile=False)
    transferred, skipped = 0, 0
    for layer in model.layers:
        try:
            weights = pretrained.get_layer(layer.name).get_weights()
            if weights:
                layer.set_weights(weights)
                transferred += 1
        except (ValueError, AttributeError):
            skipped += 1
    print(f"Transferred: {transferred} layers | Skipped: {skipped} layers")
    model = freeze_all_encoder(model)

# Compile the dataset for model training
dataset = DeepD3Dataset(
    stack_paths = TRAIN_STACKS,
    dendrite_paths = TRAIN_DENDRITES,
    spine_paths = TRAIN_SPINES,
)

In [ ]:
# Model Training - Scratch

# Only runs if if training mode is scratch
if TRAINING_MODE == 'scratch':

  # Preallocate some variables for the training process
  current_lr = INITIAL_LR
  wait = 0
  best_loss = np.inf
  ema_val_loss = None
  history = {'train_loss': [], 'val_loss': [], 'train_dice': [], 'val_dice': [], 'lr_reductions': []}

  # Sets optimizer for training and prefetches the dataset
  optimizer  = keras.optimizers.Adam(learning_rate=INITIAL_LR, clipnorm=1.0)
  prefetcher = PrefetchDataset(dataset, batch_size=BATCH_SIZE, prefetch_size=4)

  # Creates the validiation set
  val_imgs, val_d, val_s = dataset.get_batch(batch_size=BATCH_SIZE * VAL_BATCHES)

  # Training loop
  for epoch in range(EPOCHS):
      epoch_losses, epoch_dices = [], []
      t0 = time.time()

      # ------- Training -------
      for step in range(STEPS):
          # Fetches the batch from the prefetcher
          imgs, d_lbls, s_lbls = prefetcher.get_batch()

          with tf.GradientTape() as tape:
              # Has model predict and calculates the loss for dendrites and spines
              # and sums them for a total_loss
              d_pred, s_pred = model(imgs, training=True)
              d_loss = combined_loss(d_lbls, d_pred, alpha=0.5, beta=0.5)
              s_loss = combined_loss(s_lbls, s_pred, alpha=0.4, beta=0.6)
              total_loss = d_loss + s_loss

          # Backpropigation to update the model weights
          optimizer.apply_gradients(zip(tape.gradient(total_loss, model.trainable_weights),
                                        model.trainable_weights))

          # Tracks the loss and dice for this training loop
          epoch_losses.append(total_loss.numpy())
          epoch_dices.append((batch_dice(d_lbls, d_pred) + batch_dice(s_lbls, s_pred)) / 2)

      # ------- Validation -------
      v_losses, v_dices = [], []

      # Runs inference on validation set and tracks loss and dice does not update
      # the weights at all, just checking performance on held out data
      for start in range(0, len(val_imgs), BATCH_SIZE):
          vi = val_imgs[start:start+BATCH_SIZE]
          vd = val_d[start:start+BATCH_SIZE]
          vs = val_s[start:start+BATCH_SIZE]
          v_dp, v_sp = model(vi, training=False)
          v_losses.append((combined_loss(vd, v_dp, 0.5, 0.5) + combined_loss(vs, v_sp, 0.5, 0.5)).numpy())
          v_dices.append((batch_dice(vd, v_dp) + batch_dice(vs, v_sp)) / 2)

      # ------- Metrics -------
      train_loss = np.mean(epoch_losses)
      val_loss = np.mean(v_losses)
      train_dice = np.mean(epoch_dices)
      val_dice = np.mean(v_dices)

      history['train_loss'].append(train_loss);  history['val_loss'].append(val_loss)
      history['train_dice'].append(train_dice);  history['val_dice'].append(val_dice)

      # ------- Learning Rate -------

      # Applies EMA smoothing to see if learning rate needs to be adjusted
      ema_val_loss = val_loss if ema_val_loss is None else EMA_SMOOTHING * val_loss + (1 - EMA_SMOOTHING) * ema_val_loss

      # Print out metrics of this loop
      print(f"Epoch {epoch+1:2d} | Loss: {train_loss:.4f} | EMA Val: {ema_val_loss:.4f} | "
            f"Dice: {val_dice:.4f} | LR: {current_lr:.1e}", end="")

      # Adjusts learning rate of wait exceeds patience and saves the model if it
      # is the best current version of it
      if ema_val_loss < (best_loss - MIN_DELTA):
          best_loss = ema_val_loss
          wait = 0
          model.save(OUTPUT_MODEL)
          print(" SAVED", end="")
      else:
          wait += 1
          if wait >= LR_PATIENCE:
              current_lr *= LR_FACTOR
              optimizer.learning_rate.assign(current_lr)
              wait = 0
              history['lr_reductions'].append(epoch + 1)
              print(" LR REDUCED", end="")

      print(f" ({time.time()-t0:.0f}s)")

  # ------- Plotting -------
  plot_training_history(history, mode='scratch')
else:
  print("In transfer mode skipping scratch learning")

In [ ]:
# Model Training - Transfer

# Only runs if if training mode is transfer
if TRAINING_MODE == 'transfer':

  # Preallocate some variables for the transfer learning process
  current_lr = INITIAL_LR
  best_loss = np.inf
  ema_val_loss = None
  history = {'train_loss': [], 'val_loss': [], 'train_dice': [], 'val_dice': []}

  # Sets optimizer for training and prefetches the dataset
  optimizer  = keras.optimizers.Adam(learning_rate=INITIAL_LR, clipnorm=1.0)
  prefetcher = PrefetchDataset(dataset, batch_size=BATCH_SIZE, prefetch_size=4)

  # Creates the validiation set
  val_imgs, val_d, val_s = dataset.get_batch(batch_size=BATCH_SIZE * VAL_BATCHES)

  # Transfer learning loop
  for epoch in range(EPOCHS):
      epoch_losses, epoch_dices = [], []
      t0 = time.time()

      # ------- Training -------
      for step in range(STEPS):
          # Fetches the batch from the prefetcher
          imgs, d_lbls, s_lbls = prefetcher.get_batch()

          with tf.GradientTape() as tape:
              # Has model predict and calculates the loss for dendrites and spines
              # and sums them for a total_loss
              d_pred, s_pred = model(imgs, training=True)
              d_loss     = combined_loss(d_lbls, d_pred, alpha=0.5, beta=0.5)
              s_loss     = combined_loss(s_lbls, s_pred, alpha=0.4, beta=0.6)
              total_loss = d_loss + s_loss

          # Backpropigation to update the trainable model weights
          optimizer.apply_gradients(zip(tape.gradient(total_loss, model.trainable_weights),
                                        model.trainable_weights))

          # Tracks the loss and dice for this training loop
          epoch_losses.append(total_loss.numpy())
          epoch_dices.append((batch_dice(d_lbls, d_pred) + batch_dice(s_lbls, s_pred)) / 2)

      # ------- Validation -------
      v_losses, v_dices = [], []

      # Runs inference on validation set and tracks loss and dice does not update
      # the weights at all, just checking performance on held out data
      for start in range(0, len(val_imgs), BATCH_SIZE):
          vi, vd, vs = val_imgs[start:start+BATCH_SIZE], val_d[start:start+BATCH_SIZE], val_s[start:start+BATCH_SIZE]
          v_dp, v_sp = model(vi, training=False)
          v_losses.append((combined_loss(vd, v_dp, 0.5, 0.5) + combined_loss(vs, v_sp, 0.5, 0.5)).numpy())
          v_dices.append((batch_dice(vd, v_dp) + batch_dice(vs, v_sp)) / 2)

      # ------- Metrics -------
      train_loss = np.mean(epoch_losses)
      val_loss = np.mean(v_losses)
      train_dice = np.mean(epoch_dices)
      val_dice = np.mean(v_dices)

      history['train_loss'].append(train_loss);  history['val_loss'].append(val_loss)
      history['train_dice'].append(train_dice);  history['val_dice'].append(val_dice)

      # ------- Learning Rate -------

      # Applies EMA smoothing to get EMA loss
      ema_val_loss = val_loss if ema_val_loss is None else EMA_SMOOTHING * val_loss + (1 - EMA_SMOOTHING) * ema_val_loss

      print(f"Epoch {epoch+1:2d} | Loss: {train_loss:.4f} | EMA Val: {ema_val_loss:.4f} | "
            f"Dice: {val_dice:.4f} | LR: {current_lr:.1e}", end="")

      # If EMA loss is the best then it save the current model at the best interation
      if ema_val_loss < (best_loss - MIN_DELTA):
          best_loss = ema_val_loss
          model.save(OUTPUT_MODEL)
          print(" SAVED", end="")

      # ------- Progressive Unfreezing -------
      # Unfreezes next layer of encoder if it is the correct epoch based
      # on the setting in the configuration
      if epoch + 1 in UNFREEZE_SCHEDULE:
        stage, new_lr = UNFREEZE_SCHEDULE[epoch + 1]
        current_lr = new_lr
        print(f" Unfreezing stage {stage}...", end="")
        unfreeze_encoder_stage(model, optimizer, stage, new_lr)

        # Recreate optimizer with new trainable weights
        optimizer = keras.optimizers.Adam(learning_rate=new_lr, clipnorm=1.0)

      print(f" ({time.time()-t0:.0f}s)")
      print(f" ({time.time()-t0:.0f}s)")

  # ------- Plotting -------
  plot_training_history(history, mode='transfer', unfreeze_schedule=UNFREEZE_SCHEDULE)
else:
  print("In scratch mode skipping transfer learning")

In [ ]:
# Load the saved model to do some inference and post processing of the output
model = tf.keras.models.load_model(OUTPUT_MODEL, compile=False)

# Run inference using the model
stack, d_probs, s_probs, d_masks, s_masks = run_stack_inference(
    model, INFERENCE_STACK, threshold=0.35, tile_size=128, inset_size=96, batch_size=64
)

# Filter spine and dendrite segmentations using helper functions, dendrite
# filtering checks if a volume is a certain number of pixels and spine filtering
# checks that spines are close to dendrites
d_masks_cleaned  = filter_dendrite_voxels(d_masks, min_voxels=800)
s_masks_filtered = filter_spines_by_dendrite_proximity(
    d_masks_cleaned, s_masks, xy_distance_px=20, z_range=2
)

In [ ]:
# Visualize the segmentation next to the raw image
visualize_overlay(stack, d_masks_cleaned, s_masks_filtered, overlay_slice=58)

# Calculate metrics of model segmentation both with pixels and spine matching
# helper function that matches the spines via their centriods to see if they
# appear in both the real and model segmenetations
results = evaluate_stack(d_masks_cleaned, s_masks_filtered, TRUE_DENDRITE, TRUE_SPINE)
stats = evaluate_spine_centroids(
    (io.imread(TRUE_SPINE) > 0).astype(np.uint8),
    s_masks_filtered,
    pixel_resolution=(1.0, 0.096, 0.096),
    dist_threshold=2.0
)

# Print metrics
print(f"------- Centroid Matching -------")
print(f"GT spines:    {stats['true_positives'] + stats['false_negatives']}")
print(f"Detections:   {stats['true_positives'] + stats['false_positives']}")
print(f"Hits (TP):    {stats['true_positives']}")
print(f"Misses (FN):  {stats['false_negatives']}")
print(f"Ghosts (FP):  {stats['false_positives']}")
print(f"Recall:       {stats['recall']:.2%}")
print(f"Precision:    {stats['precision']:.2%}")
print(f"F1:           {stats['f1_score']:.4f}")
if not np.isnan(stats['avg_dist_microns']):
    print(f"  Avg error:    {stats['avg_dist_microns']:.3f} µm")

# Compare ground truth segmentation to model segmentation and save this image
visualize_comparison(
    stack, d_masks_cleaned, s_masks_filtered, TRUE_DENDRITE, TRUE_SPINE,
    slice_idx=58,
    # save_path=f'{OUTPUT_DIR}/spine_segmentation_comparison.pdf'
)